In [ ]:
import os
import genotools
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Release of interest
rel = 12

In [ ]:
wd = '/YOUR/WORKING/DIR'

### Generate PCs

In [ ]:
sex = 'male'
label = 'AJ'
group = f'r{rel}_v2_gp2id'

keep_file = f'{wd}/working/R{rel}_prep/imputation/{label}/{label}_release{rel}_{sex}.samples'

# Check that XY and X Regions Exist
test = pd.read_csv(f'{wd}/release/nba/raw_genotypes/{label}/GP2_{group}_post_genotools_{label}.pvar', sep = '\s+', low_memory = False)
df2 = pd.DataFrame(test['#CHROM'].value_counts()).reset_index()
chrom_list = ['X', 'Y', 'XY', 'MT']
df2 = df2[df2['#CHROM'].isin(chrom_list)]
display(df2)

In [ ]:
# Split into sex of interest
input_file = f'{wd}/release/nba/raw_genotypes/{label}/GP2_{group}_post_genotools_{label}'
output_file = f'/dev/shm/imputation/GP2_{group}_post_genotools_{label}_{sex}'

# Change directory to /dev/shm for more stoage + easy cleanup for tmp files
os.makedirs('/dev/shm/imputation', exist_ok=True)
os.chdir('/dev/shm')

! plink2 --pfile {input_file} --keep {keep_file} --make-pgen --out {output_file}

In [ ]:
# Run PCA on sex of interest
input_file = f'/dev/shm/imputation/GP2_{group}_post_genotools_{label}_{sex}'
output_file = f'{wd}/working/R{rel}_prep/XWAS/PCA/{label}_{sex}_PCA'

! genotools --pfile {input_file} --pca 15 --out {output_file}

### PCs Prep

In [ ]:
mode = 'sex_stratified'

In [ ]:
# Load in PCs
if mode == 'sex_stratified':
    eigenval_file = f"{wd}/working/R{rel}_prep/XWAS/PCA/{label}_{sex}_PCA.eigenval" # Sex-stratified results
else:
    # Using saved release-wide autosomal PCA results
    eigenval_file = f"{wd}/releases/release{rel}/raw_genotypes/{label}/{label}_release{rel}_vwb.eigenval" # Male + Female results

eigenvals = pd.read_csv(eigenval_file, header=None, names=["eigenvalue"])

In [ ]:
# Calculate % variance explained and cumulative variance
eigenvals["variance_explained"] = eigenvals["eigenvalue"] / eigenvals["eigenvalue"].sum() * 100
eigenvals["cumulative_variance"] = eigenvals["variance_explained"].cumsum()

# Find the first PC where cumulative variance >= 90
pc90 = (eigenvals["cumulative_variance"] >= 90).idxmax() + 1
var90 = eigenvals.loc[pc90 - 1, "cumulative_variance"]

# Create side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)

# Left: Variance explained
axes[0].plot(range(1, len(eigenvals) + 1), eigenvals["variance_explained"], marker="o")
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Variance Explained (%)")
axes[0].set_title("Variance Explained per PC")
axes[0].grid(True, linestyle="--", alpha=0.6)

# Right: Cumulative variance
axes[1].plot(range(1, len(eigenvals) + 1), eigenvals["cumulative_variance"], marker="s", linestyle="--")
axes[1].axhline(y=90, color="red", linestyle=":", label="90% Threshold")
axes[1].scatter(pc90, var90, color="red", zorder=5)
axes[1].text(pc90 + 0.5, var90 - 5, f"PC{pc90} ~ {var90:.1f}%", color="red")
axes[1].set_xlabel("Principal Component")
axes[1].set_ylabel("Cumulative Variance (%)")
axes[1].set_title("Cumulative Variance Explained")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.6)

fig.suptitle(f"Scree Plot: {label} R{rel}", fontsize=14, y=1.05)

plt.tight_layout()
plt.show()

### Covariate and Phenotype File Prep

In [ ]:
# Import release files
total_pheno = pd.read_csv(f'{wd}/releases/release{rel}/imputed_genotypes/{label}/chr1_{label}_release{rel}_vwb.psam', sep='\t')
total_pheno.rename(columns = {'#IID': 'IID'}, inplace = True)
display(total_pheno.head())

if mode == 'sex_stratified':
    pcs = pd.read_csv(f'{wd}/working/R{rel}_prep/XWAS/PCA/{label}_{sex}_PCA.eigenvec', sep='\t') # Sex-stratified
else:
    # Using saved release-wide autosomal PCA results
    pcs = pd.read_csv(f"{wd}/releases/release{rel}/raw_genotypes/{label}/{label}_release{rel}_vwb.eigenvec", sep='\t') # Male + Female

display(pcs.head())

change = pd.read_csv(f'{wd}/releases/release{rel}/clinical_data/r{rel}_extended_clinical_data.csv')
change = change[['GP2ID','visit_month', 'age_at_baseline', 'primary_diagnosis', 'last_diagnosis']]
display(change.head())

related = pd.read_csv(f'{wd}/releases/release{rel}/meta_data/related_samples/{label}_release{rel}_vwb.related')
display(related.head())

master = pd.read_csv(f'{wd}/releases/release{rel}/clinical_data/master_key_release{rel}_final_vwb.csv')
display(master.head())

In [ ]:
# Import release-wide psam file -- region does not matter for covariate prep
input = f'{wd}/release/nba/raw_genotypes/{label}/GP2_{group}_post_genotools_{label}'
pheno = pd.read_csv(f'{input}.psam', sep='\t')
pheno.drop(columns = 'PHENO1', inplace = True)

pheno = pheno.merge(total_pheno[['IID', 'PHENO1']], on = 'IID', how = 'inner')
display(pheno)

In [ ]:
merge1 = pd.merge(pheno, pcs, on=['#FID', 'IID'])
display(merge1)

no_dup = change.dropna(subset=['last_diagnosis'])
no_dup['diagnosis_change'] = no_dup.apply(lambda row: 'No' if row['primary_diagnosis'] == row['last_diagnosis'] else 'Yes', axis=1)
d_change = no_dup[no_dup['diagnosis_change']=='Yes']
d_change.info()

anc_d = pd.merge(d_change, merge1, left_on='GP2ID', right_on='IID')
anc_d.info()

diag_remove = anc_d['GP2ID']
anc_updated = merge1[~merge1['IID'].isin(diag_remove)]
anc_updated.info()

remove = related['IID1']
merge_norel = anc_updated[~anc_updated['IID'].isin(remove)]
merge_norel.info()

In [ ]:
pop_control = master[master['GP2_phenotype']=='Population Control']
pop_control.head()

pop_change = pop_control['GP2ID']
merge_norel['PHENO1'][merge_norel['IID'].isin(pop_change)] = 1
merge_norel.info()

merge_pheno = merge_norel[~merge_norel['PHENO1'].isnull()]
merge_pheno.info()

In [ ]:
test = pd.merge(master, merge_pheno, left_on='GP2ID', right_on='IID')
test.info()

bad = test[(test['study']=='BCM') & (test['PHENO1']==1)]
bad.head()

test = test[~test['GP2ID'].isin(bad['GP2ID'])]
test.info()

In [ ]:
pc_cols = [f'PC{i}' for i in range(1, 16)]

merge1_final = test[['#FID', 'IID', 'PHENO1', 'SEX', 'age_of_onset'] + pc_cols]
merge1_final.columns = ['#FID', 'IID', 'PHENO1', 'SEX', 'AGE'] + pc_cols
merge1_final.head()

merge1_final.fillna('-9', inplace=True)

pheno_final = merge1_final[['#FID', 'IID', 'PHENO1']]
pheno_final.rename(columns={'PHENO1': 'PHENO'}, inplace = True)

if mode == 'sex_stratified':
    merge1_final.drop(columns = ['SEX', 'AGE', 'PHENO1'], inplace = True)
else:
    # Keep SEX as a covariate when both sexes are analyzed together
    merge1_final.drop(columns = ['AGE', 'PHENO1'], inplace = True)

display(pheno_final)
display(merge1_final)

In [ ]:
pheno_final.to_csv(f'{wd}/working/R{rel}_prep/XWAS/prep_files/{sex}_{label}_pheno.txt', sep='\t', index=False)
merge1_final.to_csv(f'{wd}/working/R{rel}_prep/XWAS/prep_files/{sex}_{label}_covar.txt', sep='\t', index=False)